<a href="https://colab.research.google.com/github/GoodnessJames/Codepink/blob/master/churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

# Install the Python libraries required by the project.
!pip install -q streamlit plotly scikit-learn pandas numpy joblib

# Install LocalTunnel.
# LocalTunnel gives us a public web address that allows us
# to open the Streamlit dashboard from Google Colab.
!npm install -g localtunnel >/dev/null 2>&1

print("✅ All dependencies installed successfully.")

✅ All dependencies installed successfully.


In [22]:
# ============================================================
# CELL 2 — CREATE CUSTOMER DATA + TRAIN CHURN MODEL
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ============================================================
# 1. CREATE A REPRODUCIBLE RANDOM GENERATOR
# ============================================================

# Using 42 means the same demonstration data will be generated
# every time the notebook is run.
rng = np.random.default_rng(42)


# ============================================================
# 2. DEFINE NUMBER OF CUSTOMERS
# ============================================================

N_CUSTOMERS = 200


# ============================================================
# 3. CREATE CUSTOMER IDs
# ============================================================

customer_id = [
    f"C{1024 + i}"
    for i in range(N_CUSTOMERS)
]


# ============================================================
# 4. CREATE COMPANY NAMES
# ============================================================

company_names = [
    "Acme Ltd",
    "Beta Corp",
    "Delta Inc",
    "Gamma Ltd",
    "Omega Group",
    "Nova Systems",
    "Prime Solutions",
    "Vertex Ltd",
    "Summit Corp",
    "Atlas Inc"
]


# ============================================================
# 5. CREATE UNIQUE CUSTOMER NAMES
# ============================================================

customer = [
    f"{company_names[i % len(company_names)]} "
    f"{i // len(company_names) + 1}"
    for i in range(N_CUSTOMERS)
]


# ============================================================
# 6. CREATE CUSTOMER BEHAVIOUR VARIABLES
# ============================================================

# Average number of times a customer logs into the product
# each week.
weekly_logins = np.clip(
    rng.normal(7, 2.8, N_CUSTOMERS),
    0,
    20
)


# Percentage change in login activity.
#
# Negative number = login activity has fallen.
# Positive number = login activity has increased.
login_change_pct = np.clip(
    rng.normal(-12, 35, N_CUSTOMERS),
    -90,
    80
)


# Number of support tickets filed by the customer.
support_tickets = np.clip(
    rng.poisson(2.2, N_CUSTOMERS),
    0,
    12
)


# Percentage change in product usage.
#
# Negative number = product usage has fallen.
# Positive number = product usage has increased.
usage_change_pct = np.clip(
    rng.normal(-10, 32, N_CUSTOMERS),
    -90,
    80
)


# Number of days since the customer last logged in.
days_since_login = np.clip(
    rng.normal(6, 6, N_CUSTOMERS),
    0,
    35
)


# Approximate monthly customer spending.
monthly_spend = np.clip(
    rng.normal(180, 70, N_CUSTOMERS),
    30,
    500
)


# Number of months remaining on the customer's contract.
contract_months_left = rng.integers(
    1,
    25,
    N_CUSTOMERS
)


# ============================================================
# 7. CREATE A SYNTHETIC HISTORICAL CHURN OUTCOME
# ============================================================

# In a real company, this would come from historical data
# showing which customers actually churned.
#
# For this assignment, we create demonstration data.
#
# The formula intentionally makes certain behaviours increase
# churn likelihood:
#
# - Falling login activity
# - More support tickets
# - Falling product usage
# - More days since last login
#
# Other factors such as spending and contract duration can
# reduce the likelihood.

logit = (

    -0.85

    - 0.045 * login_change_pct

    + 0.28 * support_tickets

    - 0.035 * usage_change_pct

    + 0.16 * days_since_login

    - 0.025 * contract_months_left

    - 0.0018 * monthly_spend

    + rng.normal(
        0,
        0.75,
        N_CUSTOMERS
    )
)


# ============================================================
# 8. CONVERT THE SCORE INTO A PROBABILITY
# ============================================================

# Logistic transformation converts the model score into a
# number between 0 and 1.

probability = (
    1 /
    (1 + np.exp(-logit))
)


# ============================================================
# 9. CREATE HISTORICAL CHURN LABEL
# ============================================================

# 0 = customer did not churn
# 1 = customer churned

churned = rng.binomial(
    1,
    probability
)


# ============================================================
# 10. BUILD THE CUSTOMER DATAFRAME
# ============================================================

df = pd.DataFrame({

    "customer_id":
        customer_id,

    "customer":
        customer,

    "weekly_logins":
        weekly_logins,

    "login_change_pct":
        login_change_pct,

    "support_tickets":
        support_tickets,

    "usage_change_pct":
        usage_change_pct,

    "days_since_login":
        days_since_login,

    "monthly_spend":
        monthly_spend,

    "contract_months_left":
        contract_months_left,

    "churned":
        churned
})


# ============================================================
# 11. DEFINE MODEL FEATURES
# ============================================================

FEATURES = [

    "weekly_logins",

    "login_change_pct",

    "support_tickets",

    "usage_change_pct",

    "days_since_login",

    "monthly_spend",

    "contract_months_left"
]


# ============================================================
# 12. BUILD THE MACHINE LEARNING PIPELINE
# ============================================================

model = Pipeline([

    # StandardScaler puts the variables onto comparable scales.
    (
        "scaler",
        StandardScaler()
    ),

    # Logistic Regression predicts the probability of churn.
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])


# ============================================================
# 13. TRAIN THE MODEL
# ============================================================

model.fit(

    df[FEATURES],

    df["churned"]
)


# ============================================================
# 14. GENERATE PREDICTED CHURN PROBABILITY
# ============================================================

# [:, 1] selects the probability of class 1,
# where class 1 means "churn".

df["churn_probability"] = (

    model.predict_proba(
        df[FEATURES]
    )[:, 1]

)


# ============================================================
# 15. ASSIGN THE THREE REQUIRED RISK TIERS
# ============================================================

def get_risk_tier(probability):

    # Assignment requirement:
    # P(Churn) >= 80%
    if probability >= 0.80:

        return "High Risk"

    # Assignment requirement:
    # 40% <= P(Churn) < 80%
    elif probability >= 0.40:

        return "Medium Risk"

    # Assignment requirement:
    # P(Churn) < 40%
    else:

        return "Low Risk"


df["risk_tier"] = (

    df["churn_probability"]
    .apply(get_risk_tier)

)


# ============================================================
# 16. SAVE THE DATA
# ============================================================

# Save the customer data so the Streamlit application can
# load it later.

df.to_csv(
    "/content/customer_churn_data.csv",
    index=False
)


# ============================================================
# 17. SAVE THE TRAINED MODEL
# ============================================================

# Save the model so Streamlit can load the exact same model.

joblib.dump(
    model,
    "/content/churn_model.joblib"
)


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print("✅ Customer dataset created.")
print(
    f"✅ Number of customer accounts: {len(df)}"
)

print("\nRisk triage:")

print(
    df["risk_tier"]
    .value_counts()
    .reindex(
        [
            "High Risk",
            "Medium Risk",
            "Low Risk"
        ],
        fill_value=0
    )
)

print("\n✅ Churn model trained successfully.")

print(
    "\n✅ Saved customer data to:"
    "\n   /content/customer_churn_data.csv"
)

print(
    "\n✅ Saved trained model to:"
    "\n   /content/churn_model.joblib"
)


# Show first five customers
print("\nSample customer records:")

display(
    df.head()
)

✅ Customer dataset created.
✅ Number of customer accounts: 200

Risk triage:
risk_tier
High Risk      86
Medium Risk    65
Low Risk       49
Name: count, dtype: int64

✅ Churn model trained successfully.

✅ Saved customer data to:
   /content/customer_churn_data.csv

✅ Saved trained model to:
   /content/churn_model.joblib

Sample customer records:


,customer_id,customer,weekly_logins,login_change_pct,support_tickets,usage_change_pct,days_since_login,monthly_spend,contract_months_left,churned,churn_probability,risk_tier
0,C1024,Acme Ltd 1,7.853208,-0.184891,2,-20.778623,0.000000,247.979730,16,1,0.330432,Low Risk
1,C1025,Beta Corp 1,4.088045,37.261865,2,9.830463,13.949734,181.961553,10,0,0.433715,Medium Risk
2,C1026,Delta Inc 1,9.101263,-8.829528,0,0.876012,5.733383,164.794169,7,0,0.522374,Medium Risk
3,C1027,Gamma Ltd 1,9.633581,10.537858,0,0.113532,13.743383,165.341382,1,1,0.735261,Medium Risk
4,C1028,Omega Group 1,1.537101,-83.756024,3,3.114511,8.465895,207.337142,3,1,0.973296,High Risk


In [23]:
# ============================================================
# CELL 3 — CREATE STREAMLIT APP
# ============================================================

from pathlib import Path


# ============================================================
# CREATE THE STREAMLIT APPLICATION
# ============================================================

app_code = r"""
import streamlit as st
import pandas as pd
import plotly.express as px
import joblib


# ============================================================
# 1. PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Predictive Customer Churn Panel",
    page_icon="🎯",
    layout="wide",
    initial_sidebar_state="collapsed"
)


# ============================================================
# 2. LOAD CUSTOMER DATA
# ============================================================

df = pd.read_csv(
    "/content/customer_churn_data.csv"
)


# ============================================================
# 3. LOAD TRAINED CHURN MODEL
# ============================================================

model = joblib.load(
    "/content/churn_model.joblib"
)


# ============================================================
# 4. MODEL FEATURES
# ============================================================

FEATURES = [
    "weekly_logins",
    "login_change_pct",
    "support_tickets",
    "usage_change_pct",
    "days_since_login",
    "monthly_spend",
    "contract_months_left"
]


# ============================================================
# 5. HUMAN-READABLE FEATURE NAMES
# ============================================================

FEATURE_LABELS = {

    "weekly_logins":
        "Weekly logins",

    "login_change_pct":
        "Login activity change",

    "support_tickets":
        "Support tickets",

    "usage_change_pct":
        "Product usage change",

    "days_since_login":
        "Days since login",

    "monthly_spend":
        "Monthly spend",

    "contract_months_left":
        "Contract months left"
}


# ============================================================
# 6. CALCULATE PREDICTED CHURN PROBABILITY
# ============================================================

df["churn_probability"] = (
    model.predict_proba(
        df[FEATURES]
    )[:, 1]
)


# ============================================================
# 7. ASSIGN RISK TIER
# ============================================================

def get_risk_tier(probability):

    if probability >= 0.80:
        return "High Risk"

    elif probability >= 0.40:
        return "Medium Risk"

    else:
        return "Low Risk"


df["risk_tier"] = (
    df["churn_probability"]
    .apply(get_risk_tier)
)


# ============================================================
# 8. TOP RISK DRIVER FUNCTION
# ============================================================

def get_driver_data(row):

    scaler = model.named_steps["scaler"]

    classifier = model.named_steps["classifier"]

    x = row[
        FEATURES
    ].to_numpy(
        dtype=float
    )

    z = (
        x - scaler.mean_
    ) / scaler.scale_

    contributions = (
        z * classifier.coef_[0]
    )

    drivers = pd.DataFrame({

        "feature":
            FEATURES,

        "label":
            [
                FEATURE_LABELS[f]
                for f in FEATURES
            ],

        "value":
            x,

        "contribution":
            contributions

    })

    drivers["absolute"] = (
        drivers["contribution"]
        .abs()
    )

    return (
        drivers
        .sort_values(
            "absolute",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 9. FORMAT CUSTOMER VALUES
# ============================================================

def format_customer_value(
    feature,
    value
):

    if feature == "login_change_pct":

        arrow = "↑" if value > 0 else "↓"

        return (
            f"{arrow} {abs(value):.0f}%"
        )


    if feature == "usage_change_pct":

        arrow = "↑" if value > 0 else "↓"

        return (
            f"{arrow} {abs(value):.0f}%"
        )


    if feature == "weekly_logins":

        return (
            f"{value:.1f} / week"
        )


    if feature == "support_tickets":

        return (
            f"{value:.0f} tickets"
        )


    if feature == "days_since_login":

        return (
            f"{value:.0f} days"
        )


    if feature == "monthly_spend":

        return (
            f"${value:,.0f}"
        )


    if feature == "contract_months_left":

        return (
            f"{value:.0f} months"
        )


    return str(value)


# ============================================================
# 10. RECOMMEND AN INTERVENTION
# ============================================================

def get_recommendation(row):

    if (
        row["login_change_pct"] <= -30
        and
        row["usage_change_pct"] <= -30
    ):

        return (
            "📞 **Recommended action:** "
            "Schedule a customer-success call "
            "because both login activity and "
            "product usage have declined significantly."
        )


    if row["days_since_login"] >= 10:

        return (
            "📞 **Recommended action:** "
            "Schedule a customer-success call "
            "because the customer has been inactive "
            "for an extended period."
        )


    if row["support_tickets"] >= 5:

        return (
            "🛠️ **Recommended action:** "
            "Prioritize a support follow-up "
            "because the customer has filed "
            "multiple support tickets."
        )


    if row["usage_change_pct"] <= -30:

        return (
            "🎁 **Recommended action:** "
            "Offer a retention incentive "
            "because product usage has declined "
            "significantly."
        )


    return (
        "✉️ **Recommended action:** "
        "Send a proactive retention email "
        "to reinforce product value and engagement."
    )


# ============================================================
# 11. SESSION STATE
# ============================================================

if "actions" not in st.session_state:

    st.session_state.actions = []


def record_action(action):

    timestamp = (
        pd.Timestamp.now()
        .strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )

    st.session_state.actions.append(
        f"{timestamp} — "
        f"{selected_customer} — "
        f"{action}"
    )


# ============================================================
# 12. DASHBOARD HEADER
# ============================================================

st.title(
    "🎯 Predictive Customer Churn Panel"
)

st.caption(
    "Churn Summary — "
    "identify risky active customers → "
    "understand why → "
    "intervene immediately."
)


# ============================================================
# 13. CALCULATE RISK COUNTS
# ============================================================

high_count = int(
    (
        df["risk_tier"]
        == "High Risk"
    ).sum()
)


medium_count = int(
    (
        df["risk_tier"]
        == "Medium Risk"
    ).sum()
)


low_count = int(
    (
        df["risk_tier"]
        == "Low Risk"
    ).sum()
)


# ============================================================
# 14. RISK KPI CARDS
# ============================================================

k1, k2, k3 = st.columns(3)


with k1:

    st.metric(
        label="🔴 HIGH RISK",
        value=f"{high_count} accounts",
        delta="P(Churn) ≥ 80%",
        delta_color="inverse"
    )


with k2:

    st.metric(
        label="🟠 MEDIUM RISK",
        value=f"{medium_count} accounts",
        delta="40% ≤ P(Churn) < 80%",
        delta_color="off"
    )


with k3:

    st.metric(
        label="🟢 LOW RISK",
        value=f"{low_count} accounts",
        delta="P(Churn) < 40%",
        delta_color="normal"
    )


st.divider()


# ============================================================
# 15. RISK TRIAGE
# ============================================================

st.markdown(
    '<div class="section-title">'
    '1. RISK TRIAGE'
    '</div>',
    unsafe_allow_html=True
)


filter_col, search_col, chart_col = st.columns(
    [1, 1.5, 2]
)


# ============================================================
# 16. RISK FILTER
# ============================================================

with filter_col:

    selected_risk = st.selectbox(
        "Filter risk",
        [
            "All Risk",
            "High Risk",
            "Medium Risk",
            "Low Risk"
        ]
    )


# ============================================================
# 17. CUSTOMER SEARCH
# ============================================================

with search_col:

    search_customer = st.text_input(
        "Search customer",
        placeholder="e.g. Acme"
    )


# ============================================================
# 18. RISK DISTRIBUTION CHART
# ============================================================

with chart_col:

    chart_data = (
        df["risk_tier"]
        .value_counts()
        .reindex(
            [
                "High Risk",
                "Medium Risk",
                "Low Risk"
            ],
            fill_value=0
        )
        .reset_index()
    )

    chart_data.columns = [
        "Risk",
        "Accounts"
    ]

    fig = px.bar(
        chart_data,
        x="Risk",
        y="Accounts",
        text="Accounts",
        title="Accounts by predicted risk"
    )

    fig.update_layout(
        margin=dict(
            l=10,
            r=10,
            t=45,
            b=10
        ),
        height=230
    )

    st.plotly_chart(
        fig,
        use_container_width=True,
        config={
            "displayModeBar": False
        }
    )


# ============================================================
# 19. APPLY RISK FILTER
# ============================================================

filtered = df.copy()


if selected_risk != "All Risk":

    filtered = filtered[
        filtered["risk_tier"]
        == selected_risk
    ]


# ============================================================
# 20. APPLY CUSTOMER SEARCH
# ============================================================

if search_customer.strip():

    filtered = filtered[
        filtered["customer"].str.contains(
            search_customer.strip(),
            case=False,
            na=False
        )
    ]


# ============================================================
# 21. SORT BY CHURN PROBABILITY
# ============================================================

filtered = (
    filtered
    .sort_values(
        "churn_probability",
        ascending=False
    )
    .copy()
)


if filtered.empty:

    st.warning(
        "No customers match your filters."
    )

    st.stop()


# ============================================================
# 22. TRIAGE TABLE
# ============================================================

triage_table = filtered[
    [
        "customer",
        "churn_probability",
        "risk_tier",
        "login_change_pct",
        "usage_change_pct"
    ]
].copy()


triage_table["P(Churn)"] = (
    triage_table[
        "churn_probability"
    ]
    * 100
).round(1).astype(str) + "%"


# ============================================================
# 23. DISPLAY RISK LABEL
# ============================================================

def risk_display(value):

    if value == "High Risk":
        return "🔴 High"

    if value == "Medium Risk":
        return "🟠 Medium"

    return "🟢 Low"


triage_table["Risk"] = (
    triage_table[
        "risk_tier"
    ]
    .apply(risk_display)
)


# ============================================================
# 24. CREATE KEY SIGNAL
# ============================================================

def key_signal(row):

    login_change = abs(
        row["login_change_pct"]
    )

    usage_change = abs(
        row["usage_change_pct"]
    )


    if (
        login_change >= usage_change
        and
        login_change >= 20
    ):

        direction = (
            "↑"
            if row["login_change_pct"] > 0
            else "↓"
        )

        return (
            f"Logins {direction} "
            f"{login_change:.0f}%"
        )


    if usage_change >= 20:

        direction = (
            "↑"
            if row["usage_change_pct"] > 0
            else "↓"
        )

        return (
            f"Usage {direction} "
            f"{usage_change:.0f}%"
        )


    return "Stable"


triage_table["Key Signal"] = (
    triage_table
    .apply(
        key_signal,
        axis=1
    )
)


# ============================================================
# 25. DISPLAY TRIAGE TABLE
# ============================================================

triage_table = triage_table[
    [
        "customer",
        "P(Churn)",
        "Risk",
        "Key Signal"
    ]
]


st.dataframe(
    triage_table,
    use_container_width=True,
    hide_index=True
)


# ============================================================
# 26. SELECTED ACCOUNT
# ============================================================

st.markdown(
    '<div class="section-title">'
    '2. SELECTED ACCOUNT & TOP RISK DRIVERS'
    '</div>',
    unsafe_allow_html=True
)


customer_options = (
    filtered[
        "customer"
    ]
    .tolist()
)


selected_customer = st.selectbox(
    "Select an active customer to investigate",
    customer_options
)


customer_row = df[
    df["customer"]
    == selected_customer
].iloc[0]


probability = float(
    customer_row[
        "churn_probability"
    ]
)


risk = customer_row[
    "risk_tier"
]


# ============================================================
# 27. ACCOUNT AND DRIVER COLUMNS
# ============================================================

account_col, driver_col = st.columns(
    [1, 2]
)


# ============================================================
# 28. SELECTED ACCOUNT DETAILS
# ============================================================

with account_col:

    st.markdown(
        f"### {selected_customer}"
    )

    st.write(
        f"**Customer ID:** "
        f"{customer_row['customer_id']}"
    )

    st.metric(
        "P(Churn)",
        f"{probability * 100:.1f}%"
    )


    if risk == "High Risk":

        st.error(
            "🔴 HIGH RISK"
        )

    elif risk == "Medium Risk":

        st.warning(
            "🟠 MEDIUM RISK"
        )

    else:

        st.success(
            "🟢 LOW RISK"
        )


    st.write(
        f"**Weekly logins:** "
        f"{customer_row['weekly_logins']:.1f}"
    )

    st.write(
        f"**Login activity:** "
        f"{customer_row['login_change_pct']:+.0f}%"
    )

    st.write(
        f"**Support tickets:** "
        f"{customer_row['support_tickets']:.0f}"
    )

    st.write(
        f"**Product usage:** "
        f"{customer_row['usage_change_pct']:+.0f}%"
    )

    st.write(
        f"**Days since login:** "
        f"{customer_row['days_since_login']:.0f}"
    )


# ============================================================
# 29. TOP RISK DRIVERS
# ============================================================

with driver_col:

    st.markdown(
        "#### Top Risk Drivers"
    )


    drivers = (
        get_driver_data(
            customer_row
        )
        .head(4)
        .copy()
    )


    driver_display = []


    for rank, (_, driver) in enumerate(
        drivers.iterrows(),
        start=1
    ):

        contribution = (
            driver[
                "contribution"
            ]
        )


        if contribution > 0:

            impact = (
                "Raises churn risk"
            )

        else:

            impact = (
                "Reduces churn risk"
            )


        driver_display.append({

            "Rank":
                rank,

            "Risk Driver":
                driver["label"],

            "Customer Value":
                format_customer_value(
                    driver["feature"],
                    driver["value"]
                ),

            "Impact":
                impact,

            "Contribution":
                round(
                    contribution,
                    3
                )

        })


    driver_df = pd.DataFrame(
        driver_display
    )


    st.dataframe(
        driver_df,
        use_container_width=True,
        hide_index=True
    )



    # ========================================================
    # DRIVER CONTRIBUTION CHART
    # ========================================================

    driver_chart = (
        driver_df
        .sort_values(
            "Contribution",
            ascending=True
        )
    )


    fig2 = px.bar(
        driver_chart,
        x="Contribution",
        y="Risk Driver",
        orientation="h",
        title="Estimated driver contribution"
    )


    fig2.add_vline(
        x=0,
        line_width=1
    )


    fig2.update_layout(
        margin=dict(
            l=10,
            r=10,
            t=45,
            b=10
        ),
        height=300
    )


    st.plotly_chart(
        fig2,
        use_container_width=True,
        config={
            "displayModeBar": False
        }
    )


st.divider()


# ============================================================
# 30. SUGGESTED INTERVENTIONS
# ============================================================

st.header("3. SUGGESTED INTERVENTIONS")


recommendation = get_recommendation(
    customer_row
)


st.info(
    recommendation
)


# ============================================================
# 31. INTERVENTION BUTTONS
# ============================================================

b1, b2, b3, b4 = st.columns(4)


# ============================================================
# 32. DISCOUNT
# ============================================================

with b1:

    if st.button(
        "🎁 Trigger 20% Code",
        use_container_width=True
    ):

        record_action(
            "20% retention discount code triggered"
        )

        st.success(
            "20% retention discount code triggered."
        )


# ============================================================
# 33. SCHEDULE CALL
# ============================================================

with b2:

    if st.button(
        "📞 Schedule Call",
        use_container_width=True
    ):

        record_action(
            "Customer-success call scheduled"
        )

        st.success(
            "Customer-success call scheduled."
        )


# ============================================================
# 34. SEND EMAIL
# ============================================================

with b3:

    if st.button(
        "✉️ Send Retention Email",
        use_container_width=True
    ):

        record_action(
            "Retention email sent"
        )

        st.success(
            "Retention email sent."
        )


# ============================================================
# 35. CREATE FOLLOW-UP TASK
# ============================================================

with b4:

    if st.button(
        "📋 Create Follow-up Task",
        use_container_width=True
    ):

        record_action(
            "Follow-up task created"
        )

        st.success(
            "Follow-up task created."
        )


# ============================================================
# 36. ACTIVITY LOG
# ============================================================

if st.session_state.actions:

    with st.expander(
        "Recent intervention activity",
        expanded=False
    ):

        for action in reversed(
            st.session_state.actions[-10:]
        ):

            st.write(
                f"• {action}"
            )


# ============================================================
# WRITE THE STREAMLIT CODE TO app.py
# ============================================================

app_path = Path(
    "/content/app.py"
)


app_path.write_text(
    app_code,
    encoding="utf-8"
)


print(
    "✅ Streamlit application created successfully."
)

print(
    "File created:"
)

print(
    "/content/app.py"
)
"""

In [24]:
# ============================================================
# KILL ALL OLD PROCESSES
# ============================================================
!pkill -f streamlit
!pkill -f cloudflared

In [25]:
# ============================================================
# CELL 4 — LAUNCH STREAMLIT DASHBOARD
# ============================================================

import subprocess
import time
import re

# Start Streamlit
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give Streamlit time to start
time.sleep(5)

# Install Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# Start Cloudflare Tunnel
cloudflare_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for the public URL
public_url = None

for _ in range(30):

    line = cloudflare_process.stdout.readline()

    if "trycloudflare.com" in line:

        match = re.search(
            r"https://[-a-zA-Z0-9]+\.trycloudflare\.com",
            line
        )

        if match:

            public_url = match.group(0)

            break


if public_url:

    print("=" * 70)
    print("🌐 PREDICTIVE CUSTOMER CHURN PANEL")
    print("=" * 70)
    print()
    print("Open this URL in your browser:")
    print()
    print(public_url)
    print()
    print("=" * 70)

else:

    print(
        "The tunnel URL was not detected automatically."
    )

    print(
        "Check the output above for a trycloudflare.com URL."
    )

🌐 PREDICTIVE CUSTOMER CHURN PANEL

Open this URL in your browser:

https://converter-executives-newspaper-applies.trycloudflare.com

